Data-Splitting

In [ ]:
# Prepare the data
tsla_close = tsla_data[['Adj Close']].copy()

# Split the data
train_size = int(len(tsla_close) * 0.8)
train_data = tsla_close.iloc[:train_size]
test_data = tsla_close.iloc[train_size:]

ARIMA Model

In [ ]:
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Find the best ARIMA model
auto_model = auto_arima(train_data, seasonal=False, stepwise=True, suppress_warnings=True, trace=True)
print(auto_model.summary())

# Fit the model and make predictions
predictions_arima = auto_model.predict(n_periods=len(test_data))
predictions_arima = pd.Series(predictions_arima, index=test_data.index)

# Evaluate the model
mae_arima = mean_absolute_error(test_data, predictions_arima)
rmse_arima = np.sqrt(mean_squared_error(test_data, predictions_arima))
mape_arima = np.mean(np.abs((test_data.values - predictions_arima.values) / test_data.values)) * 100

print(f"ARIMA MAE: {mae_arima:.4f}")
print(f"ARIMA RMSE: {rmse_arima:.4f}")
print(f"ARIMA MAPE: {mape_arima:.4f}%")

LSTM Model

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

# Scale the data
scaler = MinMaxScaler()
scaled_train_data = scaler.fit_transform(train_data)

# Create sequences
def create_dataset(dataset, time_step=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-time_step-1):
        a = dataset[i:(i+time_step), 0]
        dataX.append(a)
        dataY.append(dataset[i + time_step, 0])
    return np.array(dataX), np.array(dataY)

time_step = 60
X_train, y_train = create_dataset(scaled_train_data, time_step)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

# Build the LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(time_step, 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
model.fit(X_train, y_train, batch_size=1, epochs=1)

# Make predictions
inputs = tsla_close[len(tsla_close) - len(test_data) - time_step:].values
inputs = inputs.reshape(-1,1)
inputs = scaler.transform(inputs)

X_test = []
for i in range(time_step, len(inputs)):
    X_test.append(inputs[i-time_step:i, 0])
X_test = np.array(X_test)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

predictions_lstm = model.predict(X_test)
predictions_lstm = scaler.inverse_transform(predictions_lstm)

# Evaluate the model
mae_lstm = mean_absolute_error(test_data, predictions_lstm)
rmse_lstm = np.sqrt(mean_squared_error(test_data, predictions_lstm))
mape_lstm = np.mean(np.abs((test_data.values - predictions_lstm) / test_data.values)) * 100

print(f"LSTM MAE: {mae_lstm:.4f}")
print(f"LSTM RMSE: {rmse_lstm:.4f}")
print(f"LSTM MAPE: {mape_lstm:.4f}%")

Model Comparison

In [ ]:
# Create a DataFrame for comparison
metrics = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE (%)'],
    'ARIMA': [mae_arima, rmse_arima, mape_arima],
    'LSTM': [mae_lstm, rmse_lstm, mape_lstm]
})
print(metrics)

# Plot the predictions
plt.figure(figsize=(14, 7))
plt.plot(train_data, label='Train Data')
plt.plot(test_data, label='Actual Prices')
plt.plot(predictions_arima, label='ARIMA Predictions')
plt.plot(test_data.index, predictions_lstm, label='LSTM Predictions')
plt.title('TSLA Stock Price Prediction')
plt.xlabel('Date')
plt.ylabel('Adjusted Close Price (USD)')
plt.legend()
plt.grid(True)
plt.savefig('../results/plots/model_comparison.png')
plt.show()